In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv
/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/LCDataDictionary.xlsx


In [2]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [3]:
data = pd.read_csv('/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv')
data.head()

/tmp/ipykernel_16/4254957762.py:1: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/kaggle/input/datasets/adarshsng/lending-club-loan-data-csv/loan.csv')


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
member_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,2260668.0,15046.931228,9190.245488,500.00,8000.00,12900.000,20000.0000,40000.00
funded_amnt,2260668.0,15041.664057,9188.413022,500.00,8000.00,12875.000,20000.0000,40000.00
funded_amnt_inv,2260668.0,15023.437624,9192.331807,0.00,8000.00,12800.000,20000.0000,40000.00
...,...,...,...,...,...,...,...,...
hardship_payoff_balance_amount,10613.0,11628.036442,7615.161123,55.73,5628.73,10044.220,16114.9400,40306.41
hardship_last_payment_amount,10613.0,193.606331,198.694368,0.01,43.78,132.890,284.1800,1407.86
settlement_amount,33056.0,5030.606922,3692.027842,44.21,2227.00,4172.855,6870.7825,33601.00
settlement_percentage,33056.0,47.775600,7.336379,0.20,45.00,45.000,50.0000,521.35


# PREPROCESSING

## Target Encoding

In [5]:
# Filter the Target Variable (loan_status)
# We only want loans that have finished their lifecycle.
valid_statuses = ['Fully Paid', 'Charged Off']
data = data[data['loan_status'].isin(valid_statuses)].copy()

# Create the binary target label
# 1 = Default (Charged Off), 0 = Good (Fully Paid)
data['default'] = np.where(data['loan_status'] == 'Charged Off', 1, 0)

## Numerical Cleaning

1. **Dropping NaN(s)**: Drop columns with almost every element as NaN.
2. **Dropping Cheater Features**: Drop columns that causes leakages in our pipeline.
3. **Fill in remaining**: Fill remaining NaNs with some values for modeling.

### NaN Checks

In [6]:
# Check the NaN percentage for every column
missing_fractions = data.isnull().mean().sort_values(ascending=False)
print("Top 20 columns with highest NaN percentages:")
print(missing_fractions.head(20))

Top 20 columns with highest NaN percentages:
id                                            1.000000
member_id                                     1.000000
next_pymnt_d                                  1.000000
url                                           1.000000
orig_projected_additional_accrued_interest    0.997367
hardship_end_date                             0.995908
hardship_loan_status                          0.995908
hardship_start_date                           0.995908
hardship_dpd                                  0.995908
hardship_type                                 0.995908
deferral_term                                 0.995908
hardship_amount                               0.995908
hardship_status                               0.995908
hardship_reason                               0.995908
hardship_length                               0.995908
hardship_payoff_balance_amount                0.995908
hardship_last_payment_amount                  0.995908
payment_plan_start_d

In [7]:
# ---------------------------------------------------------
# Drop High-NaN Columns
# ---------------------------------------------------------

# If a column is missing more than 30% of its data, it's generally 
# too sparse to be reliable for our ML model. We drop them.
threshold = 0.30 
# thresh requires the minimum number of NON-NaN values to keep the column
data = data.dropna(thresh=len(data) * (1 - threshold), axis=1)
print(f"Shape after dropping high-NaN columns: {data.shape}")

Shape after dropping high-NaN columns: (1303607, 88)


### Dropping Cheaters

In [8]:
# ---------------------------------------------------------
# Drop "Cheater" (Data Leakage) Columns
# ---------------------------------------------------------

# These are columns that contain information generated AFTER the loan was issued 
# (like recoveries, settlement amounts, and total payments). 
# We also drop 'loan_status' because it's what we built our 'default' target from!

leakage_cols = [
    'loan_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt',
    'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'debt_settlement_flag', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage',
    'settlement_term', 'hardship_flag', 'hardship_type', 
    'hardship_reason', 'hardship_status', 'deferphal_term',
    'payment_plan_start_date', 'orig_projected_additional_accrued_interest'
]

# We only drop columns that survived Step 1
cols_to_drop = [col for col in leakage_cols if col in data.columns]
data = data.drop(columns=cols_to_drop)
print(f"Shape after dropping leakage columns: {data.shape}")

Shape after dropping leakage columns: (1303607, 73)


### Fill Remaining

In [9]:
# ---------------------------------------------------------
# Impute (Fill in) Remaining Missing Values
# ---------------------------------------------------------
# The ML model cannot handle any remaining NaNs. 

# A. For numeric columns, we fill missing values with the median of that column.
# (Median is better than Mean because it isn't skewed by crazy outliers)
num_cols = data.select_dtypes(include=['float64', 'int64']).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# B. For text/categorical columns, we just replace missing data with the word 'Unknown'.
cat_cols = data.select_dtypes(include=['object']).columns
data[cat_cols] = data[cat_cols].fillna('Unknown')

# Verify we have zero NaNs left
total_nans = data.isnull().sum().sum()
print(f"Total NaNs remaining in dataset: {total_nans}")

Total NaNs remaining in dataset: 0


## Categorical Encoding

In [10]:
# ---------------------------------------------------------
# Format and Encode Categorical Variables
# ---------------------------------------------------------

print(f"Shape before encoding: {data.shape}")

# A. Extract numbers from text-heavy columns
if 'term' in data.columns:
    # Converts " 36 months" to 36
    data['term'] = data['term'].str.extract(r'(\d+)').astype(float)

if 'emp_length' in data.columns:
    # Convert employment length to numbers (e.g., "10+ years" -> 10, "< 1 year" -> 0)
    # We mapped 'Unknown' to NaNs earlier, so we handle that too.
    emp_map = {
        '10+ years': 10, '9 years': 9, '8 years': 8, '7 years': 7,
        '6 years': 6, '5 years': 5, '4 years': 4, '3 years': 3,
        '2 years': 2, '1 year': 1, '< 1 year': 0, 'Unknown': 0
    }
    data['emp_length'] = data['emp_length'].map(emp_map).fillna(0)

# B. Drop useless text columns
# 'emp_title' and 'title' have hundreds of thousands of unique, messy strings (e.g., "Registered Nurse", "RN", "Nurse"). 
# 'zip_code' is redacted (e.g., "123xx"). 
useless_text_cols = ['emp_title', 'title', 'zip_code', 'url', 'desc']
cols_to_drop = [col for col in useless_text_cols if col in data.columns]
data = data.drop(columns=cols_to_drop)

# --- Drop High-Cardinality Text Columns (Like Dates and IDs) ---
# We find all text columns, count their unique values, and drop those with > 40 unique values.
# This automatically handles messy date strings and states.
categorical_cols = data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if data[col].nunique() > 40:
        print(f"Dropping {col} due to high cardinality ({data[col].nunique()} unique values)")
        data = data.drop(columns=[col])

# C. Label Encode remaining categorical columns
# We assign a unique integer to each category (e.g., Rent = 0, Mortgage = 1, Own = 2)

le = LabelEncoder()

categorical_cols = data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    data[col] = le.fit_transform(data[col].astype(str))

print(f"Final shape ready for Machine Learning: {data.shape}")

Shape before encoding: (1303607, 73)
Dropping issue_d due to high cardinality (139 unique values)
Dropping addr_state due to high cardinality (51 unique values)
Dropping earliest_cr_line due to high cardinality (738 unique values)
Final shape ready for Machine Learning: (1303607, 67)


In [11]:
print("----Final Dataset for Modeling----")
data.head()

----Final Dataset for Modeling----


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,default
100,30000,30000,30000.0,36.0,22.35,1151.16,3,19,5,1,...,89.5,33.3,1.0,0.0,527120.0,98453.0,28600.0,101984.0,0,0
152,40000,40000,40000.0,60.0,16.14,975.71,2,13,0,1,...,100.0,42.9,0.0,0.0,344802.0,161720.0,45700.0,167965.0,0,0
170,20000,20000,20000.0,36.0,7.56,622.68,0,2,10,1,...,94.7,20.0,0.0,0.0,622183.0,71569.0,85100.0,74833.0,0,0
186,4500,4500,4500.0,36.0,11.31,147.99,1,7,10,5,...,91.7,0.0,0.0,0.0,53795.0,29137.0,15100.0,24595.0,0,0
215,8425,8425,8425.0,36.0,27.27,345.18,4,24,3,1,...,100.0,50.0,0.0,0.0,768304.0,189194.0,45800.0,189054.0,0,0


# MODEL BUILDING

## Splitting Dataset

In [12]:
print(f"Dataset shape before splitting: {data.shape}")
# ---------------------------------------------------------
# STEP 1: Define Features (X) and Target (y)
# ---------------------------------------------------------
# We drop 'default' from X because it is the answer key.
X = data.drop(columns=['default'])
y = data['default']

# ---------------------------------------------------------
# STEP 2: Train/Test Split
# ---------------------------------------------------------
# We split the data: 80% for training the model, 20% for testing its accuracy.
# stratify=y ensures that the 80/20 split maintains the exact same ratio of defaults.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# ---------------------------------------------------------
# STEP 3: Handle Class Imbalance
# ---------------------------------------------------------
# There are far more "Good" loans (0) than "Default" loans (1).
# We calculate the exact ratio so XGBoost pays more attention to the defaults.
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()
imbalance_ratio = num_negative / num_positive

print(f"Class Imbalance Ratio (Good / Default): {imbalance_ratio:.2f}")

Dataset shape before splitting: (1303607, 67)
Training data shape: (1042885, 66)
Testing data shape: (260722, 66)
Class Imbalance Ratio (Good / Default): 3.98


## XGB model training

In [13]:
# ---------------------------------------------------------
# STEP 4: Initialize and Train XGBoost Classifier
# ---------------------------------------------------------
print("Training XGBoost Model... (This may take a minute or two)")

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,            # Number of trees
    max_depth=7,                 # How deep each tree can go (prevents overfitting)
    learning_rate=0.03,           # How aggressively it learns
    scale_pos_weight=imbalance_ratio, # Crucial: Punishes the model harder for missing defaults
    eval_metric='auc',           # Optimize for Area Under the ROC Curve
    random_state=42,
    n_jobs=-1                    # Use all available CPU cores
)

# Fit the model to the training data
xgb_model.fit(X_train, y_train)
print("Training Complete!")

# ---------------------------------------------------------
# STEP 5: Evaluate Model Performance
# ---------------------------------------------------------
# We generate raw probabilities (e.g., 0.15 chance of default)
# and hard predictions (0 or 1 based on a 0.5 threshold)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = xgb_model.predict(X_test)

# ROC-AUC is the industry standard for credit risk. 
# 0.5 is a random coin flip. 0.7+ is good. 0.8+ is excellent.
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"\n--- Model Evaluation ---")
print(f"ROC-AUC Score: {auc_score:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training XGBoost Model... (This may take a minute or two)
Training Complete!

--- Model Evaluation ---
ROC-AUC Score: 0.7315

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.67      0.76    208391
           1       0.34      0.67      0.45     52331

    accuracy                           0.67    260722
   macro avg       0.61      0.67      0.60    260722
weighted avg       0.78      0.67      0.70    260722

Confusion Matrix:
[[139337  69054]
 [ 17428  34903]]


In [14]:
# ---------------------------------------------------------
# STEP 6: Save the Model and Feature List
# ---------------------------------------------------------
# We save the trained model so we can load it later in our Dash App.
# We also save the exact column names so our app knows what inputs the model expects.

joblib.dump(xgb_model, 'pd_model.pkl')
joblib.dump(X_train.columns.tolist(), 'model_features.pkl')

print("\nModel saved as 'pd_model.pkl'")
print("Feature list saved as 'model_features.pkl'")


Model saved as 'pd_model.pkl'
Feature list saved as 'model_features.pkl'


In [15]:
sample_portfolio = data.sample(n=20, random_state=42)
sample_portfolio.to_csv('live_portfolio.csv', index=False)